In [1]:
# Imports and paths

from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

RAW_FILE = (
    PROJECT_ROOT
    / "Raw_Data"
    / "NFHS_5_India_Districts_Factsheet_Data.xls"
)

STAGING_DIR = PROJECT_ROOT / "Staging"
REPORT_DIR = (
    PROJECT_ROOT
    / "Reports"
    / "02_staging_and_profiling"
)

STAGING_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

STAGING_FILE = STAGING_DIR / "nfhs_staging.csv"

print("Project paths configured.")
print(f"Raw file: {RAW_FILE}")
print(f"Staging file: {STAGING_FILE}")
print(f"Reports folder: {REPORT_DIR}")

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Raw dataset not found: {RAW_FILE}")

Project paths configured.
Raw file: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Raw_Data\NFHS_5_India_Districts_Factsheet_Data.xls
Staging file: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Staging\nfhs_staging.csv
Reports folder: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\02_staging_and_profiling


In [2]:
# Load source data and standardise columns

df_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Sheet1",
    engine="xlrd"
)

def standardize_column_name(column):
    """Convert a column name to lowercase snake_case."""
    column = re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        str(column).strip().lower()
    )
    return column.strip("_")


df_staging = df_raw.copy()

# Standardise names and make duplicate column names unique.
new_columns = []
seen = {}

for column in df_staging.columns:
    name = standardize_column_name(column)
    count = seen.get(name, 0)

    new_columns.append(
        name if count == 0 else f"{name}_{count}"
    )
    seen[name] = count + 1

df_staging.columns = new_columns

# Validate required identifier columns.
required_columns = {"district_names", "state_ut"}
missing_columns = required_columns - set(df_staging.columns)

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {sorted(missing_columns)}"
    )

# Trim identifier text without changing indicator values.
for column in ["district_names", "state_ut"]:
    df_staging[column] = (
        df_staging[column]
        .astype("string")
        .str.strip()
    )

print(f"Source rows: {len(df_raw):,}")
print(f"Staging rows: {len(df_staging):,}")
print(f"Staging columns: {len(df_staging.columns):,}")

assert df_staging.shape == df_raw.shape, (
    "Staging row/column structure differs from source."
)

Source rows: 707
Staging rows: 707
Staging columns: 109


In [3]:
# Save staging data

df_staging.to_csv(
    STAGING_FILE,
    index=False
)

# Verify the saved file.
df_staging_check = pd.read_csv(
    STAGING_FILE,
    low_memory=False
)

assert df_staging_check.shape == df_staging.shape, (
    "Saved staging file shape does not match the DataFrame."
)

print(f"Staging file saved and verified: {STAGING_FILE}")
print(f"Rows: {len(df_staging_check):,}")
print(f"Columns: {len(df_staging_check.columns):,}")

Staging file saved and verified: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Staging\nfhs_staging.csv
Rows: 707
Columns: 109


In [4]:
# Profile staging data

profile = pd.DataFrame({
    "column_name": df_staging.columns,
    "data_type": df_staging.dtypes.astype(str).values,
    "missing_values": df_staging.isna().sum().values,
    "missing_percentage": (
        df_staging.isna().mean().mul(100).round(2).values
    ),
    "unique_values": df_staging.nunique().values,
})

profile_file = REPORT_DIR / "staging_data_profile.csv"

profile.to_csv(
    profile_file,
    index=False
)

print(f"Total missing values: {df_staging.isna().sum().sum():,}")
print(f"Duplicate rows: {df_staging.duplicated().sum():,}")
print(
    "Duplicate district/state combinations:",
    df_staging.duplicated(
        subset=["district_names", "state_ut"]
    ).sum()
)

display(profile)

print(f"Profile report saved: {profile_file}")

Total missing values: 0
Duplicate rows: 0
Duplicate district/state combinations: 0


,column_name,data_type,missing_values,missing_percentage,unique_values
0,district_names,string,0,0.0,699
1,state_ut,string,0,0.0,36
2,number_of_households_surveyed,int64,0,0.0,185
3,number_of_women_age_15_49_years_interviewed,int64,0,0.0,435
4,number_of_men_age_15_54_years_interviewed,int64,0,0.0,146
...,...,...,...,...,...
104,women_age_30_49_years_ever_undergone_an_oral_c...,float64,0,0.0,183
105,women_age_15_years_and_above_who_use_any_kind_...,float64,0,0.0,605
106,men_age_15_years_and_above_who_use_any_kind_of...,float64,0,0.0,657
107,women_age_15_years_and_above_who_consume_alcohol,float64,0,0.0,295


Profile report saved: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\02_staging_and_profiling\staging_data_profile.csv


In [5]:
# District and state name quality checks

# Work on a copy; preserve the staging dataset.
district_audit = df_staging[
    ["district_names", "state_ut"]
].copy()

# Normalize whitespace for comparisons only.
for column in ["district_names", "state_ut"]:
    district_audit[column] = (
        district_audit[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# 1. Missing or blank district/state names.
missing_names = district_audit[
    district_audit["district_names"].isna()
    | district_audit["district_names"].eq("")
    | district_audit["state_ut"].isna()
    | district_audit["state_ut"].eq("")
].copy()

# 2. Duplicate district-state combinations.
duplicate_pairs = district_audit[
    district_audit.duplicated(
        subset=["district_names", "state_ut"],
        keep=False
    )
].sort_values(["state_ut", "district_names"])

# 3. District names occurring in multiple states.
district_state_conflicts = (
    district_audit
    .dropna(subset=["district_names", "state_ut"])
    .groupby("district_names")["state_ut"]
    .nunique()
    .loc[lambda counts: counts > 1]
    .rename("state_count")
    .reset_index()
)

# Save quality reports.
missing_names.to_csv(
    REPORT_DIR / "district_state_missing_names.csv",
    index=False
)

duplicate_pairs.to_csv(
    REPORT_DIR / "duplicate_district_state_pairs.csv",
    index=False
)

district_state_conflicts.to_csv(
    REPORT_DIR / "district_names_multiple_states.csv",
    index=False
)

print("Missing district/state names:", len(missing_names))
print("Rows in duplicate district-state pairs:", len(duplicate_pairs))
print(
    "District names occurring in multiple states:",
    len(district_state_conflicts)
)

display(district_state_conflicts)

Missing district/state names: 0
Rows in duplicate district-state pairs: 0
District names occurring in multiple states: 8


,district_names,state_count
0,Aurangabad,2
1,Balrampur,2
2,Bijapur,2
3,Bilaspur,2
4,Chandel,2
5,Hamirpur,2
6,Pratapgarh,2
7,Raigarh,2


In [6]:
# Inspect district names that occur in multiple states

conflict_names = district_state_conflicts["district_names"].tolist()

conflict_details = (
    district_audit[
        district_audit["district_names"].isin(conflict_names)
    ]
    .drop_duplicates()
    .sort_values(["district_names", "state_ut"])
    .reset_index(drop=True)
)

display(conflict_details)

,district_names,state_ut
0,Aurangabad,Bihar
1,Aurangabad,Maharastra
2,Balrampur,Chhattisgarh
3,Balrampur,Uttar Pradesh
4,Bijapur,Chhattisgarh
5,Bijapur,Karnataka
6,Bilaspur,Chhattisgarh
7,Bilaspur,Himachal Pradesh
8,Chandel,Manipur
9,Chandel,Mizoram


In [7]:
# Review potential naming and geographic issues

# Inspect the state/UT combinations for repeated district names.
conflict_names = district_state_conflicts["district_names"].tolist()

conflict_details = (
    district_audit[
        district_audit["district_names"].isin(conflict_names)
    ]
    .drop_duplicates()
    .sort_values(["district_names", "state_ut"])
    .reset_index(drop=True)
)

display(conflict_details)

conflict_details.to_csv(
    REPORT_DIR / "district_state_conflict_details.csv",
    index=False
)

# Record the proposed spelling correction for review.
correction_log = pd.DataFrame([
    {
        "column": "state_ut",
        "source_value": "Maharastra",
        "proposed_value": "Maharashtra",
        "reason": "State name spelling",
        "status": "Proposed correction - verify before applying"
    }
])

correction_log.to_csv(
    REPORT_DIR / "proposed_name_corrections.csv",
    index=False
)

# Record the geographic pairing that needs checking.
geographic_review = pd.DataFrame([
    {
        "district_names": "Chandel",
        "state_ut": "Mizoram",
        "issue": "District-state pairing requires verification",
        "action": "Check source row against the original factsheet"
    }
])

geographic_review.to_csv(
    REPORT_DIR / "district_state_review_required.csv",
    index=False
)

# Review all state/UT names and their row counts.
state_name_review = (
    district_audit["state_ut"]
    .value_counts(dropna=False)
    .rename_axis("state_ut")
    .reset_index(name="row_count")
    .sort_values("state_ut", na_position="last")
    .reset_index(drop=True)
)

display(correction_log)
display(geographic_review)
display(state_name_review)

correction_log.to_csv(
    REPORT_DIR / "proposed_name_corrections.csv",
    index=False
)

state_name_review.to_csv(
    REPORT_DIR / "state_name_review.csv",
    index=False
)

print(f"All Notebook 2 reports saved under: {REPORT_DIR}")

,district_names,state_ut
0,Aurangabad,Bihar
1,Aurangabad,Maharastra
2,Balrampur,Chhattisgarh
3,Balrampur,Uttar Pradesh
4,Bijapur,Chhattisgarh
5,Bijapur,Karnataka
6,Bilaspur,Chhattisgarh
7,Bilaspur,Himachal Pradesh
8,Chandel,Manipur
9,Chandel,Mizoram


,column,source_value,proposed_value,reason,status
0,state_ut,Maharastra,Maharashtra,State name spelling,Proposed correction - verify before applying


,district_names,state_ut,issue,action
0,Chandel,Mizoram,District-state pairing requires verification,Check source row against the original factsheet


,state_ut,row_count
0,Andaman & Nicobar Islands,3
1,Andhra Pradesh,13
2,Arunachal Pradesh,20
3,Assam,33
4,Bihar,38
5,Chandigarh,1
6,Chhattisgarh,27
7,Dadra and Nagar Haveli & Daman and Diu,3
8,Goa,2
9,Gujarat,33


All Notebook 2 reports saved under: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\02_staging_and_profiling
